## Print information about a tuned chorale
1. Chorale name, tolerance, tonal diamond shape, limit max
1. Cent values, note names, scores, and ratios for every chord
2. Top notes cents, note names, cent values


In [18]:
import os
local_dir = os.getcwd()  # Current working directory
print(f'The local directory is: {local_dir}')

The local directory is: /home/prent/Repos/One-footed-bride-tuning


In [19]:
import logging, os, sys, time
from importlib import reload
import numpy as np
from importlib import reload
from collections import Counter, defaultdict
user = 'prent'
base_dir = local_dir
WAVE_DIR = os.path.join(base_dir, 'Music', 'sflib')
numpy_dir = os.path.join(base_dir, 'Archive','opt')
np.set_printoptions(legacy='1.25')

import diamond_music_utils as dmu
import adaptive_tuning_util as atu 
from itertools import count, combinations, permutations
dmu.start_logger('test.log',log_level = 'info') # how to modify this so that it only prints to the log and not in the notebook.
logging.info(f'{base_dir = }, {numpy_dir = }, {WAVE_DIR = }')
rng = np.random.default_rng()

In [20]:
def print_chords(version, input_file, numpy_dir, measure, tolerance, ratios=True, print_individual_chords=True,\
            offset=0, use_werck_top_notes=False, print_top_notes = True):
    
    _, _, chorale, root, mode, keys = atu.load_chorale_in_cents(version, numpy_dir)
    try:
        # input_file = os.path.join(numpy_dir, f'{version}-cents.npy')
        floating_cents = np.load(input_file)
        existing_chorale_in_cents = np.rint(floating_cents).astype(int)
        # print(f'loaded cents file from {input_file}')
    except:
        print(f'Trouble loading {input_file = }')
        return {input_file}
    if use_werck_top_notes:
        input_file = os.path.join(numpy_dir, f'{version}-w-top_notes.npy')
    else: 
        input_file = os.path.join(numpy_dir, f'{version}top-notes.npy')
    try:      
        top_notes = np.load(input_file)
    except:
        print(f'Trouble loading {input_file = }')
        return {input_file}
    top_notes[1] = top_notes[1] + offset
    
    if print_top_notes:
        print(f'Key: {keys[root]} {mode}, {tolerance = }')
        print(f'\ntop notes:')
        print(*[inx for inx in np.arange(12)], sep='\t')
        print(*[note for note in top_notes[0]], sep = '\t')
        print(*[keys[note] for note in top_notes[0]], sep = '\t')
        print(*[cent_value for cent_value in top_notes[1]], sep = '\t')
    if print_individual_chords: 
        print(f'\n#          cents       note names   chord score')
        # #     +----- cents -----+--- note names---+--- chord score'
        # 0:    0  386    0  884	C♮ E♮ C♮ A♮	47.0
    if measure > 0: print(f'\nprinting only measure {measure}')
    prev_chord = np.zeros(4, dtype=int)
    header1 = f" # Fr/To Cents Ratio\t # Fr/To Cents Ratio\t # Fr/To Cents Ratio"
    
    for inx, chord_in_cents, chord_12 in zip(count(0,1), existing_chorale_in_cents.T, chorale.T):
        if not np.array_equal(prev_chord, chord_in_cents):
            if measure == 0 or 16 * (measure - 1) <= inx < 16 * measure:
                if print_individual_chords: 
                        # Join the note names into a single space-separated string to avoid numpy array formatting
                        pitches = ' '.join(map(str, keys[chord_12 % 12]))
                        print(f'{inx}: {atu.format_chord(chord_in_cents,4)}\t{pitches}\t{chord_scorer.score_chord(chord_in_cents, tolerance=tolerance)}')
                if ratios:
                    print(f'{header1}')
                    intervals = []
                    for inx1, inx2 in combinations(np.arange(4),2):
                            cent_value_interval_pair = np.array([chord_in_cents[inx1], chord_in_cents[inx2]])
                            cent_value_delta, cent_value_moves, cent_value_target = atu.cent_value_interval(cent_value_interval_pair)
                            best_idx = chord_scorer.find_best_interval(cent_value_delta, tolerance)[0]
                            ratio = str(atu.limit_format(tonal_diamond[best_idx])[0]).strip()
                            n1 = keys[chord_12[inx1] % 12]
                            n2 = keys[chord_12[inx2] % 12]
                            intervals.append((n1, n2, cent_value_delta, ratio))

                    def fmt(iv, idx):
                            n1, n2, cents, ratio = iv
                            return f"{idx:>2} {n1:>2} {n2:>2} {cents:>5} {ratio:^6}"

                    # print first and last three intervals on separate lines, nicely aligned and without Python punctuation
                    
                    print("   ".join(fmt(iv, i+1) for i, iv in enumerate(intervals[:3])))
                    print("   ".join(fmt(iv, i+1+3) for i, iv in enumerate(intervals[3:])))
        prev_chord = chord_in_cents.copy()
    return keys, root, mode

In [21]:
reload(atu)

limit_max = 19
measure = 0 # 0 means print all measures
print_individual_chords = False
use_werck_top_notes = False
ratios = False
print_top_notes = False
print_hits_misses = False
# Archive/opt/bwv253-trans-sa-opt.npy
total_scores = 0
num_scores = 0
max_score = 0
tonal_diamond = atu.build_tonal_diamond(limit_max)
chord_scorer = atu.ChordScorer(tonal_diamond)
chord_scorer.reset_cache()
for tolerance in [1,2,3,4]:
    # The latest files are here: Archive/opt/bwv253-trans-sa-opt.npy
    print(f'{numpy_dir = }, {tolerance = }, {limit_max = }')
    print(f'{tonal_diamond.shape = }, {measure = }')
    for version in ['bwv253', 'bwv254', 'bwv255', 'bwv256', 'bwv257', 'bwv258', 'bwv259', 'bwv260',  'bwv261', 'bwv262', 'bwv263', 'bwv264']: # ['bwv256']: #
        try:
            input_file = os.path.join(numpy_dir, f'tolerance-{tolerance}', f'{version}-trans-sa-opt.npy')
            # input_file = os.path.join(numpy_dir, f'{version}-cents.npy')
            existing_chorale_in_cents = np.load(input_file)
            logging.info(f'{input_file = }')
        except:
            print(f'Trouble loading {input_file = }')
            continue
        num_scores += 1
        scores = np.array([chord_scorer.score_chord(chord, tolerance=tolerance) for chord in existing_chorale_in_cents.T])
        print(f'\nversion: {version}, Tol: {tolerance}, Average score: {round(np.average(scores),1)}, max score: {np.max(scores)} max chord: {np.argmax(scores)}')
        total_scores += np.average(scores)
        max_score = np.max([max_score, np.max(scores) ])
        keys, root, mode = print_chords(version, input_file, numpy_dir, measure, tolerance, ratios=ratios, print_individual_chords=print_individual_chords, use_werck_top_notes=use_werck_top_notes, print_top_notes = print_top_notes)
        if print_hits_misses:
            print(f'hits and misses: {chord_scorer.return_cache_results()}')
        print(f'overall total: {round(total_scores,1)}, {num_scores = }, Average Score: {round(np.average(total_scores/num_scores),1)}, {max_score = }')

numpy_dir = '/home/prent/Repos/One-footed-bride-tuning/Archive/opt', tolerance = 1, limit_max = 19
tonal_diamond.shape = (84, 3), measure = 0

version: bwv253, Tol: 1, Average score: 48.8, max score: 90.0 max chord: 122
overall total: 48.8, num_scores = 1, Average Score: 48.8, max_score = 90.0

version: bwv254, Tol: 1, Average score: 54.6, max score: 140.0 max chord: 158
overall total: 103.5, num_scores = 2, Average Score: 51.7, max_score = 140.0

version: bwv255, Tol: 1, Average score: 50.7, max score: 82.0 max chord: 42
overall total: 154.2, num_scores = 3, Average Score: 51.4, max_score = 140.0

version: bwv256, Tol: 1, Average score: 53.9, max score: 140.0 max chord: 138
overall total: 208.1, num_scores = 4, Average Score: 52.0, max_score = 140.0

version: bwv257, Tol: 1, Average score: 56.9, max score: 140.0 max chord: 128
overall total: 264.9, num_scores = 5, Average Score: 53.0, max_score = 140.0

version: bwv258, Tol: 1, Average score: 57.5, max score: 140.0 max chord: 80
overa